# 01 · Data Loading & Cleaning

Loads four macroeconomic series pulled from **FRED** (Federal Reserve Economic Data):

| Series | Meaning | Frequency |
|---|---|---|
| `CPIAUCSL` | Consumer Price Index (inflation) | Monthly |
| `UNRATE` | Unemployment rate | Monthly |
| `FEDFUNDS` | Federal Funds effective rate | Monthly |
| `SP500` | S&P 500 index level | Daily |

Builds two clean, analysis-ready datasets and explains an important data
limitation discovered while building this project (see the note below).

In [1]:
import pandas as pd
import os

RAW = "../Data/raw/"
PROC = "../Data/processed/"
os.makedirs(PROC, exist_ok=True)

cpi = pd.read_csv(RAW + "CPIAUCSL.csv", parse_dates=["observation_date"])
unrate = pd.read_csv(RAW + "UNRATE.csv", parse_dates=["observation_date"])
fedfunds = pd.read_csv(RAW + "FEDFUNDS.csv", parse_dates=["observation_date"])
sp500 = pd.read_csv(RAW + "SP500.csv", parse_dates=["observation_date"])

for name, df in [("CPI", cpi), ("UNRATE", unrate), ("FEDFUNDS", fedfunds), ("SP500", sp500)]:
    print(f"{name:9s} {df['observation_date'].min().date()} -> {df['observation_date'].max().date()}  ({len(df)} rows)")

CPI       1947-01-01 -> 2025-12-01  (948 rows)
UNRATE    1948-01-01 -> 2025-12-01  (936 rows)
FEDFUNDS  1954-07-01 -> 2026-01-01  (859 rows)
SP500     2021-02-04 -> 2026-02-04  (1305 rows)


### ⚠️ Data limitation worth calling out

`SP500` starts in **2021**, while `CPIAUCSL`, `UNRATE`, and `FEDFUNDS` go back to the
1950s. This isn't a bug in the download — FRED's `SP500` series is licensed from
S&P and only keeps a rolling ~5-year window of history.

That means: **any dataset that includes SP500 will be limited to ~2021-onward**,
no matter how far back the other three series go. So this notebook builds
**two** processed datasets instead of one:

1. `macro_long_history.csv` — CPI, unemployment, Fed Funds only, full history back to 1954 (~70 years)
2. `master_monthly_dataset.csv` — all four series together, necessarily limited to ~2021-2025

This matters a lot downstream: it means CPI/unemployment/rate models can be trained on
decades of data, but anything involving SP500 (including SP500's own forecast) is working
with a much thinner, more recent-history-biased sample — flagged again in notebook 03.

In [2]:
# A couple of raw files have a single NaN row for a month FRED hadn't published yet
# (e.g. a government-shutdown-delayed release) -- drop those rather than let them
# propagate into merges.
cpi = cpi.dropna(subset=["CPIAUCSL"])
unrate = unrate.dropna(subset=["UNRATE"])
fedfunds = fedfunds.dropna(subset=["FEDFUNDS"])

# SP500 is daily -> resample to a monthly average to match the others
sp500_monthly = (
    sp500.set_index("observation_date")
    .resample("MS")
    .mean()
    .reset_index()
    .dropna(subset=["SP500"])
)
print("SP500 monthly shape:", sp500_monthly.shape)
sp500_monthly.head()

SP500 monthly shape: (61, 2)


,observation_date,SP500
0,2021-02-01,3897.179375
1,2021-03-01,3910.508261
2,2021-04-01,4141.176190
3,2021-05-01,4167.849500
4,2021-06-01,4238.489545


In [3]:
# Long-history macro dataset: CPI + UNRATE + FEDFUNDS share history back to 1954
macro = (
    cpi.merge(unrate, on="observation_date", how="inner")
       .merge(fedfunds, on="observation_date", how="inner")
       .sort_values("observation_date")
       .dropna()
)
macro.to_csv(PROC + "macro_long_history.csv", index=False)
print("macro_long_history:", macro.shape, macro.observation_date.min().date(), "->", macro.observation_date.max().date())
macro.head()

macro_long_history: (857, 4) 1954-07-01 -> 2025-12-01


,observation_date,CPIAUCSL,UNRATE,FEDFUNDS
0,1954-07-01,26.86,5.8,0.80
1,1954-08-01,26.85,6.0,1.22
2,1954-09-01,26.81,6.1,1.07
3,1954-10-01,26.72,5.7,0.85
4,1954-11-01,26.78,5.3,0.83


In [4]:
# Full dataset including SP500 -- inner join necessarily truncates to SP500's window
full = macro.merge(sp500_monthly, on="observation_date", how="inner").sort_values("observation_date")
full.to_csv(PROC + "master_monthly_dataset.csv", index=False)
print("master_monthly_dataset:", full.shape, full.observation_date.min().date(), "->", full.observation_date.max().date())
full.head()

master_monthly_dataset: (58, 5) 2021-02-01 -> 2025-12-01


,observation_date,CPIAUCSL,UNRATE,FEDFUNDS,SP500
0,2021-02-01,263.573,6.2,0.08,3897.179375
1,2021-03-01,264.847,6.1,0.07,3910.508261
2,2021-04-01,266.625,6.1,0.07,4141.176190
3,2021-05-01,268.404,5.8,0.06,4167.849500
4,2021-06-01,270.710,5.9,0.08,4238.489545


### Summary

- `macro_long_history.csv` — 857 rows, **1954-07 to 2025-12** (CPI, UNRATE, FEDFUNDS)
- `master_monthly_dataset.csv` — 58 rows, **2021-02 to 2025-12** (adds SP500)

Both are saved to `Data/processed/` and used by the next two notebooks.